# PhysioMI L/R: facial motor cortex electrode ablation (Table A7, 20 runs per condition)

Self-contained reproduction of the electrode ablation of Appendix E.3 / Table A7 with finetuned LaBraM on PhysioMI left/right,
using a symmetric facial motor cortex (MC) set L = {T9, T10, T7, T8, FT7, FT8, FC5, FC6, C5, C6}
and 20 runs per condition (the paper's table: 5 runs, asymmetric set).

| | All available electrodes | Facial MC electrodes only | Without facial MC electrodes |
|---|---|---|---|
| Condition name | baseline | only_lateral | holdout_lateral |
| Input | all 64 channels | the 10 channels of L | 54 channels (L dropped) |
| Seed indices *i* (seed = 42 + *i*) | 0–19 | 20–39 | 20–39 |
| Model, data, split, training | identical (`src/config/production/physionet_finetuned.yaml`) | identical | identical |

**Test (the only one):** two-sided Mann–Whitney U on per-run test balanced accuracy, "All available electrodes" vs
"Without facial MC electrodes", *n* = 20 vs 20 (normal approximation, tie correction, no continuity correction;
reported in APA style with *U*, *z*, *p* and the effect size *r*, section 5). Being the single test, it needs no multiple-testing correction.
The "Facial MC electrodes only" row is descriptive.

**How to run.**
* `RUN_TRAINING = False` (default) only recomputes the test from the finished runs in `OUTPUT_DIR`.
* `RUN_TRAINING = True` trains and tests every run that is missing from `OUTPUT_DIR`. It needs a kernel on a GPU node (one APU, about 10–20 min per run, 60 runs). Point `OUTPUT_DIR` to an empty folder to retrain everything from scratch.
  Each run is one `src/main.py` call (section 1); the original runs were launched as one SLURM job per run with the same call.
  GPU training with mixed precision is not bit-deterministic, so retrained accuracies will vary slightly between runs.

In [1]:
import glob
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# Repository root (folder holding pyproject.toml), independent of where the kernel was started.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SRC_DIR = PROJECT_ROOT / "src"

# ---- Experiment definition ----
BASE_CONFIG = "src/config/production/physionet_finetuned.yaml"
SEED_BASE = 42
LATERAL_ELECTRODES = "T9,T10,T7,T8,FT7,FT8,FC5,FC6,C5,C6"
# condition -> (table label, seed indices, electrodes to drop, electrodes to keep exclusively, run LRP after testing)
CONDITIONS = {
    "baseline": ("All available electrodes", range(0, 20), None, None, True),
    "only_lateral": ("Facial MC electrodes only", range(20, 40), None, LATERAL_ELECTRODES, False),
    "holdout_lateral": ("Without facial MC electrodes", range(20, 40), LATERAL_ELECTRODES, None, False),
}

# ---- Paths and switches ----
DATA_DIR = "/ptmp/$USER/physiomi_lr"  # full PhysioMI extraction: files/S###/S###R##.edf
OUTPUT_DIR = "/ptmp/$USER/logs/clever_hans_suite_confirmatory"  # <OUTPUT_DIR>/<condition>/run_<i>/
NUM_WORKERS = 24  # dataloader workers (= SLURM --cpus-per-task of the original runs)
RUN_TRAINING = False

print(PROJECT_ROOT)

/viper/u1/$USER/projects/foundational_xai_in_neuroimaging


## Statistical test and reporting helpers

All group comparisons in this notebook use the **Mann–Whitney U test** (`scipy.stats.mannwhitneyu`) with
**normal approximation, tie correction and no continuity correction**.
Without ties this yields exactly the z and p of `scipy.stats.ranksums`, which earlier versions of the analysis used.

Every test reports, in APA style:
* each group characterized by *n*, median (Mdn) and interquartile range (IQR = Q1–Q3),
* *U* (statistic of the first-named group: number of pairs in which its value is larger, ties counting ½),
* *z* (tie-corrected; *z* > 0 means the first-named group tends to be larger),
* *p* (and the FDR-adjusted *p* where several tests are run),
* the effect size *r* = *z* / √*N*, and *A* = *U* / (*n*₁*n*₂), the probability that a random value of the first group exceeds one of the second.

The cell also defines the booktabs (LaTeX) table builder and `write_output`, which saves the `.tex` tables and a
`.json` statistics cache (inputs, results, code and library versions, data locations) to `results/apa_mwu/`.
This cell is identical in the four notebooks that report these statistics (listed at the top of the cell).

In [2]:
# ---- Mann–Whitney U test with APA reporting, booktabs tables and a statistics cache ----
# This cell is identical in quantifying_attr_on_arousal.ipynb, quantifying_attr_on_physiomi.ipynb,
# clever_hans_lateral_holdout_mwu.ipynb and electrode_statistics_summary.ipynb.
import datetime
import json
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import mannwhitneyu, norm

# Repository root (folder holding pyproject.toml), independent of where the kernel was started.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STATS_DIR = PROJECT_ROOT / "results" / "apa_mwu"  # .tex tables and .json statistics caches are written here

# Normal approximation with tie correction, without continuity correction.
# Without ties this gives exactly the z and p of scipy.stats.ranksums (checked below where ranksums was used before).
MWU_SETTINGS = {"method": "asymptotic", "use_continuity": False}
MWU_SETTINGS_TEXT = "normal approximation with tie correction, no continuity correction"


def mwu_apa(x, y, alternative, label_x, label_y):
    """Mann–Whitney U test of x vs. y, returning every quantity needed for APA reporting.

    U    statistic of x: number of pairs (x_i, y_j) with x_i > y_j, ties counting 1/2 (scipy's convention).
    z    (U - n_x n_y / 2) / sigma_U with the tie-corrected sigma_U; z > 0 means x tends to be larger than y.
    p    from z under the given alternative ("greater": x tends to be larger than y).
    r    effect size z / sqrt(n_x + n_y).
    A    U / (n_x n_y), the probability that a random x exceeds a random y (common-language effect size).
    Mdn, Q1, Q3 describe each group (IQR = Q1–Q3, linear interpolation).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_x, n_y = len(x), len(y)
    n = n_x + n_y
    res = mannwhitneyu(x, y, alternative=alternative, **MWU_SETTINGS)
    u = float(res.statistic)
    _, tie_sizes = np.unique(np.concatenate([x, y]), return_counts=True)
    sigma_u = np.sqrt(n_x * n_y / 12 * ((n + 1) - np.sum(tie_sizes**3 - tie_sizes) / (n * (n - 1))))
    z = (u - n_x * n_y / 2) / sigma_u
    # scipy does not return z; the p-value recomputed from our z must equal scipy's p-value
    p_from_z = {"greater": norm.sf(z), "less": norm.cdf(z), "two-sided": 2 * norm.sf(abs(z))}[alternative]
    assert np.isclose(p_from_z, res.pvalue, rtol=1e-9, atol=0), (p_from_z, res.pvalue)
    (q1_x, mdn_x, q3_x), (q1_y, mdn_y, q3_y) = np.percentile(x, [25, 50, 75]), np.percentile(y, [25, 50, 75])
    return {
        "label_x": label_x, "label_y": label_y, "alternative": alternative,
        "n_x": n_x, "mdn_x": mdn_x, "q1_x": q1_x, "q3_x": q3_x,
        "n_y": n_y, "mdn_y": mdn_y, "q1_y": q1_y, "q3_y": q3_y,
        "U": u, "z": z, "p": float(res.pvalue), "r": z / np.sqrt(n), "A": u / (n_x * n_y),
        "n_tied_values": int(np.sum(tie_sizes[tie_sizes > 1])),  # observations sharing their value with another one
    }


# ---- APA number formatting ----
def fmt_bounded(v, digits=2):
    """Quantities that cannot exceed 1 in absolute value (p, r, rho): no leading zero."""
    return f"{v:.{digits}f}".replace("0.", ".", 1)


def fmt_p(p):
    """'< .001', '> .999' or '= .023'."""
    if p < 0.001:
        return "< .001"
    if p > 0.999:
        return "> .999"
    return "= " + fmt_bounded(p, 3)


def fmt_u(u):
    return f"{u:.1f}".removesuffix(".0")  # U is an integer or ends in .5 (ties)


def describe_group(s, which, digits=3):
    """Characterization of one group, e.g. 'relevance at AF7 (n = 15, Mdn = .041, IQR = .027–.052)'.

    All compared quantities here (relative relevance, balanced accuracy) cannot exceed 1: no leading zero (APA).
    """
    f = lambda v: fmt_bounded(v, digits)
    return (f"{s['label_' + which]} (n = {s['n_' + which]}, Mdn = {f(s['mdn_' + which])}, "
            f"IQR = {f(s['q1_' + which])}–{f(s['q3_' + which])})")


def apa_mwu_text(s, p_adj=None):
    """'U = 11204, z = 4.12, p < .001, r = .14' (with 'p_FDR ...' if an adjusted p-value is given)."""
    text = f"U = {fmt_u(s['U'])}, z = {s['z']:.2f}, p {fmt_p(s['p'])}"
    if p_adj is not None:
        text += f", p_FDR {fmt_p(p_adj)}"
    return text + f", r = {fmt_bounded(s['r'])}"


def apa_mwu_sentence(s, alpha=0.05, p_adj=None, digits=3, describe_y=True):
    """Full sentence: both groups characterized, the direction, and the test statistics.

    Significance is decided on p_adj if given, otherwise on p.
    describe_y=False names the second group without n, Mdn and IQR, for a group that is characterized once
    for many tests (see describe_background).
    """
    significant = (p_adj if p_adj is not None else s["p"]) < alpha
    if s["alternative"] == "greater":
        verdict = "was significantly higher than" if significant else "was not significantly higher than"
    elif s["alternative"] == "less":
        verdict = "was significantly lower than" if significant else "was not significantly lower than"
    elif significant:
        verdict = "was significantly higher than" if s["z"] > 0 else "was significantly lower than"
    else:
        verdict = "did not differ significantly from"
    first = describe_group(s, "x", digits)
    second = describe_group(s, "y", digits) if describe_y else s["label_y"]
    return f"{first[0].upper()}{first[1:]} {verdict} {second}, {apa_mwu_text(s, p_adj)}."


def fmt_range(values, digits=3, dash="–"):
    """'.014' if all values round to the same number, else 'min–max', e.g. '.024–.025'."""
    lo, hi = fmt_bounded(min(values), digits), fmt_bounded(max(values), digits)
    return lo if lo == hi else f"{lo}{dash}{hi}"


def describe_background(results, digits=3, tex=False):
    """One phrase characterizing the background group of all electrode-vs-background tests at once.

    The background of electrode c is all other electrodes, so it hardly changes from test to test. Its n, Mdn, Q1 and
    Q3 are given as ranges over the tests (one value where all tests round to the same number); "virtually identical"
    is only stated if every range spans at most one unit of the last digit shown. The per-test values stay in the
    results (n_y, mdn_y, q1_y, q3_y).
    """
    dash = "--" if tex else "–"
    stats = ", ".join(f"{name} = {fmt_range(results[col], digits, dash)}" for name, col in [("Mdn", "mdn_y"), ("Q1", "q1_y"), ("Q3", "q3_y")])
    n_y = sorted(set(results["n_y"]))
    n_text = str(n_y[0]) if len(n_y) == 1 else f"{n_y[0]}{dash}{n_y[-1]}"
    n_text = f"$n = {n_text}$" if tex else f"n = {n_text}"
    identical = all(results[col].max() - results[col].min() <= 10 ** -digits for col in ["mdn_y", "q1_y", "q3_y"])
    verdict = "was virtually identical in all" if identical else "varied across the"
    return (f"The background group, the relative relevance of all other electrodes in all runs ({n_text} per test), "
            f"{verdict} {len(results)} tests ({stats} across tests).")


# ---- LaTeX (booktabs) output; needs \usepackage{booktabs} (and \usepackage{longtable} for long=True) ----
def tex_p(p):
    return "$< .001$" if p < 0.001 else ("$> .999$" if p > 0.999 else f"${fmt_bounded(p, 3)}$")


def tex_num(v, digits=2, bounded=False):
    return f"${fmt_bounded(v, digits) if bounded else f'{v:.{digits}f}'}$"  # math mode gives a proper minus sign


def booktabs(header, rows, colspec, caption, label, long=False):
    """booktabs table in \\small with narrower column spacing.

    header: list of cells (one header row), or list of rows, where a row is a list of cells or a raw line (e.g. a \\cmidrule).
    long=False: floating table. long=True: longtable that breaks across pages (header repeated), for full electrode lists.
    An empty table gets one row saying so.
    """
    n_cols = sum(ch in "lcr" for ch in colspec)
    rows_of_header = header if isinstance(header[0], list) else [header]
    head = "\n".join(r if isinstance(r, str) else " & ".join(r) + r" \\" for r in rows_of_header)
    body = [" & ".join(row) + r" \\" for row in rows] or [rf"\multicolumn{{{n_cols}}}{{l}}{{No electrode with $p_{{\mathrm{{FDR}}}} < .05$}} \\"]
    if long:
        return "\n".join([
            r"{\small\setlength{\tabcolsep}{4pt}",
            rf"\begin{{longtable}}{{{colspec}}}",
            rf"\caption{{{caption}}}\label{{{label}}} \\",
            r"\toprule", head, r"\midrule", r"\endfirsthead",
            rf"\multicolumn{{{n_cols}}}{{l}}{{\emph{{(continued)}}}} \\",
            r"\toprule", head, r"\midrule", r"\endhead",
            r"\bottomrule", r"\endfoot",
            *body,
            r"\end{longtable}",
            r"}",
        ])
    return "\n".join([
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
        head,
        r"\midrule",
        *body,
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])


# ---- Output files ----
def _git(*args):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()


def provenance(data_sources):
    """Everything needed to trace the numbers back: code version, library versions, data locations, test settings."""
    return {
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "git_commit": _git("rev-parse", "HEAD"),
        "git_branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
        "git_uncommitted_changes": bool(_git("status", "--porcelain")),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "mwu_settings": {**MWU_SETTINGS, "tie_correction": True, "description": MWU_SETTINGS_TEXT},
        "data_sources": data_sources,
    }


def _jsonable(o):
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (pd.DataFrame, pd.Series)):
        return json.loads(o.to_json(orient="records" if isinstance(o, pd.DataFrame) else "index"))
    if isinstance(o, Path):
        return str(o)
    raise TypeError(type(o))


def write_output(filename, content):
    """Write a .tex table (str) or a .json statistics cache (dict) to STATS_DIR and print the path."""
    STATS_DIR.mkdir(parents=True, exist_ok=True)
    path = STATS_DIR / filename
    if isinstance(content, str):
        path.write_text(content + "\n")
    else:
        path.write_text(json.dumps(content, indent=1, default=_jsonable, ensure_ascii=False) + "\n")
    print(f"written: {path}")

## 1. The `src/main.py` call of every run

Every run calls the existing pipeline with this argument list;
average reference, filtering, epoching and scaling happen inside the PhysioMI datamodule as in the paper,
and `--ablation_strategy drop` removes the excluded (or all non-included) channels from the input sequence
(the remaining channels keep their pretrained LaBraM position embeddings).

In [3]:
def run_dir(condition, seed_idx):
    return os.path.join(OUTPUT_DIR, condition, f"run_{seed_idx}")


def main_argv(condition, seed_idx):
    _, _, exclude, include, attribute = CONDITIONS[condition]
    argv = [
        "--config", BASE_CONFIG,
        "--data_dir", DATA_DIR,
        "--output_dir", run_dir(condition, seed_idx),
        "--seed", str(SEED_BASE + seed_idx),
        "--num_workers", str(NUM_WORKERS),
        "--do_testing",
        "--ablation_strategy", "drop",
    ]
    if attribute:
        argv.append("--attribute")
    if exclude:
        argv.extend(["--exclude_electrodes", exclude])
    if include:
        argv.extend(["--include_electrodes", include])
    return argv


for condition, (_, seeds, _, _, _) in CONDITIONS.items():
    print(f"{condition}, seed index {seeds[0]}:")
    print("  python src/main.py " + " ".join(main_argv(condition, seeds[0])) + "\n")

baseline, seed index 0:
  python src/main.py --config src/config/production/physionet_finetuned.yaml --data_dir /ptmp/$USER/physiomi_lr --output_dir /ptmp/$USER/logs/clever_hans_suite_confirmatory/baseline/run_0 --seed 42 --num_workers 24 --do_testing --ablation_strategy drop --attribute

only_lateral, seed index 20:
  python src/main.py --config src/config/production/physionet_finetuned.yaml --data_dir /ptmp/$USER/physiomi_lr --output_dir /ptmp/$USER/logs/clever_hans_suite_confirmatory/only_lateral/run_20 --seed 62 --num_workers 24 --do_testing --ablation_strategy drop --include_electrodes T9,T10,T7,T8,FT7,FT8,FC5,FC6,C5,C6

holdout_lateral, seed index 20:
  python src/main.py --config src/config/production/physionet_finetuned.yaml --data_dir /ptmp/$USER/physiomi_lr --output_dir /ptmp/$USER/logs/clever_hans_suite_confirmatory/holdout_lateral/run_20 --seed 62 --num_workers 24 --do_testing --ablation_strategy drop --exclude_electrodes T9,T10,T7,T8,FT7,FT8,FC5,FC6,C5,C6



## 2. Training and testing (only if `RUN_TRAINING`)

Each run trains with early stopping on validation loss and evaluates the checkpoint with the best
validation balanced accuracy once on the 10 test subjects, writing `test_results.json`.
Runs that already have `test_results.json` are skipped.

The environment settings below were applied to the original runs; none of them changes the method:
* `HF_DATASETS_CACHE`: preprocessed datasets cache shared by both conditions (both use the raw data).
* ROCm (AMD MI300A) only: route GEMMs to rocBLAS instead of hipBLASLt (speed only).
* LRP attribution of the baseline runs runs in-process instead of in a worker pool (speed only; identical results).
  It runs after testing and does not affect `test_results.json`.

In [4]:
def is_tested(condition, seed_idx):
    return bool(glob.glob(os.path.join(run_dir(condition, seed_idx), "*", "version_*", "test_results.json")))


if RUN_TRAINING:
    hf_home = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
    os.environ["HF_DATASETS_CACHE"] = os.path.join(hf_home, "datasets_clever_hans", "raw")

    import torch
    if torch.version.hip:
        torch.backends.cuda.preferred_blas_library("cublas")
        os.environ["DISABLE_ADDMM_CUDA_LT"] = "1"

    sys.path.insert(0, str(SRC_DIR))
    os.chdir(PROJECT_ROOT)  # main.py resolves the config and checkpoint paths relative to the repository
    import main as main_module

    attribute_dataset_on_labram = main_module.attribute_dataset_on_labram

    def attribute_sequentially(*a, **kw):
        kw["num_workers"] = 0
        return attribute_dataset_on_labram(*a, **kw)

    main_module.attribute_dataset_on_labram = attribute_sequentially

    for condition, (_, seeds, _, _, _) in CONDITIONS.items():
        for seed_idx in seeds:
            if is_tested(condition, seed_idx):
                print(f"{condition} run {seed_idx}: already tested, skipped")
                continue
            print(f"{condition} run {seed_idx}: training")
            main_module.main(main_argv(condition, seed_idx))

## 3. Per-run test results

In [5]:
rows = []
for condition, (_, seeds, _, _, _) in CONDITIONS.items():
    for seed_idx in seeds:
        paths = glob.glob(os.path.join(run_dir(condition, seed_idx), "*", "version_*", "test_results.json"))
        assert len(paths) == 1, f"{condition} run {seed_idx}: expected one test_results.json, found {paths}"
        with open(paths[0]) as f:
            res = json.load(f)
        rows.append({"condition": condition, "seed_idx": seed_idx, "seed": SEED_BASE + seed_idx,
                     "AUROC": res["test_auroc"], "Bal. Acc.": res["test_balanced_accuracy"], "F1": res["test_f1"]})
runs = pd.DataFrame(rows)

pd.concat([runs[runs.condition == c].reset_index(drop=True)[["seed", "Bal. Acc."]].round(3) for c in CONDITIONS],
          axis=1, keys=list(CONDITIONS))

baseline           only_lateral           holdout_lateral          
       seed Bal. Acc.         seed Bal. Acc.            seed Bal. Acc.
0        42     0.789           62     0.751              62     0.524
1        43     0.761           63     0.757              63     0.755
2        44     0.518           64     0.758              64     0.515
3        45     0.573           65     0.523              65     0.515
4        46     0.627           66     0.753              66     0.769
5        47     0.765           67     0.746              67     0.512
6        48     0.749           68     0.490              68     0.574
7        49     0.752           69     0.564              69     0.646
8        50     0.740           70     0.493              70     0.498
9        51     0.733           71     0.742              71     0.769
10       52     0.560           72     0.519              72     0.522
11       53     0.776           73     0.543              73     0.514
12       54     0.506           74     0.492              74     0.508
13       55     0.780           75     0.763              75     0.547
14       56     0.533           76     0.738              76     0.597
15       57     0.494           77     0.759              77     0.760
16       58     0.534           78     0.560              78     0.531
17       59     0.548           79     0.724              79     0.577
18       60     0.770           80     0.469              80     0.515
19       61     0.778           81     0.550              81     0.528

## 4. Table A7 with the new runs

Mean (std.) across runs of the test-set metrics; std. is the sample standard deviation (ddof = 1).

In [6]:
metrics = ["AUROC", "Bal. Acc.", "F1"]
grouped = runs.groupby("condition", sort=False)[metrics]
table_a7 = (grouped.mean().map("{:.3f}".format) + " (" + grouped.std().map("{:.3f}".format) + ")")
table_a7.insert(0, "n", grouped.size())
table_a7.index = [CONDITIONS[c][0] for c in table_a7.index]
table_a7.index.name = "Condition"
table_a7

,n,AUROC,Bal. Acc.,F1
Condition,,,,
All available electrodes,20,0.725 (0.149),0.664 (0.116),0.663 (0.109)
Facial MC electrodes only,20,0.688 (0.165),0.635 (0.120),0.622 (0.123)
Without facial MC electrodes,20,0.612 (0.129),0.584 (0.099),0.587 (0.100)


## 5. Mann–Whitney U test: all available electrodes vs. without facial MC electrodes (two-sided)

Settings and output as described in the helper section at the top. *U* is the statistic of the runs with all available
electrodes; *A* = *U* / (*n*₁*n*₂) is the probability that a random run with all electrodes has a higher balanced
accuracy than a random run without facial MC electrodes.

**Change from the earlier version of this notebook:** the test was run with scipy's defaults, which apply a continuity
correction; now it is run without (as `scipy.stats.ranksums` and SPSS do), so that all tests in the paper use the same
settings. The cell prints both p-values.

In [7]:
from scipy.stats import ranksums

baseline = runs.loc[runs.condition == "baseline", "Bal. Acc."].to_numpy()
holdout = runs.loc[runs.condition == "holdout_lateral", "Bal. Acc."].to_numpy()

mwu = mwu_apa(baseline, holdout, "two-sided",
              label_x="test balanced accuracy with all available electrodes",
              label_y="test balanced accuracy without facial MC electrodes")
print(apa_mwu_sentence(mwu))
print(f"\nU = {fmt_u(mwu['U'])}, z = {mwu['z']:.4f}, p = {mwu['p']:.4f}, r = {mwu['r']:.4f}, "
      f"A = P(baseline run > holdout run) = {mwu['A']:.2f}, tied values: {mwu['n_tied_values']}")

# Comparison with the earlier call (scipy defaults: continuity correction) and with scipy.stats.ranksums
previous = mannwhitneyu(baseline, holdout, alternative="two-sided")
rs = ranksums(baseline, holdout)
comparison = {
    "previous_call": "mannwhitneyu(baseline, holdout, alternative='two-sided') with scipy defaults",
    "previous_method": "exact" if min(len(baseline), len(holdout)) <= 8 and mwu["n_tied_values"] == 0 else "asymptotic with continuity correction",
    "previous_U": float(previous.statistic), "previous_p": float(previous.pvalue),
    "ranksums_z": float(rs.statistic), "ranksums_p": float(rs.pvalue),
}
print(f"earlier version:  U = {previous.statistic:.0f}, p = {previous.pvalue:.4f} ({comparison['previous_method']})")
print(f"ranksums:         z = {rs.statistic:.4f}, p = {rs.pvalue:.4f} (no tie correction; equal to the above if no ties)")

Test balanced accuracy with all available electrodes (n = 20, Mdn = .736, IQR = .545–.767) was significantly higher than test balanced accuracy without facial MC electrodes (n = 20, Mdn = .530, IQR = .515–.609), U = 281, z = 2.19, p = .028, r = .35.

U = 281, z = 2.1911, p = 0.0284, r = 0.3464, A = P(baseline run > holdout run) = 0.70, tied values: 0
earlier version:  U = 281, p = 0.0294 (asymptotic with continuity correction)
ranksums:         z = 2.1911, p = 0.0284 (no tie correction; equal to the above if no ties)


**Note on the pre-specified analysis.** In the statistics plan fixed on 2026-10-27, before the confirmatory runs,
this comparison was one of four primary comparisons with Holm correction; there its adjusted p-value is twice the
p-value above. This notebook reports it as the single test.

## 6. LaTeX table (booktabs)

Requires `\usepackage{booktabs}` in the preamble. Also written to `results/apa_mwu/table_a7.tex`.

In [8]:
n_runs = sorted(set(table_a7["n"]))
n_text = f"{n_runs[0]} runs per condition" if len(n_runs) == 1 else "runs per condition: " + ", ".join(map(str, table_a7["n"]))
lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Model performance across " + n_text + r" based on electrode inclusion. Notation: Mean (std.). "
    r"Two-sided Mann--Whitney $U$ test on balanced accuracy, all available electrodes vs.\ without facial MC "
    rf"electrodes ({MWU_SETTINGS_TEXT}): $U = {fmt_u(mwu['U'])}$, $z = {mwu['z']:.2f}$, $p {fmt_p(mwu['p'])}$, "
    rf"$r = {fmt_bounded(mwu['r'])}$.}}",
    r"\label{tab:facial_mc_ablation}",
    r"\begin{tabular}{lccc}",
    r"\toprule",
    r"Condition & AUROC & Bal.\ Acc. & F1 \\",
    r"\midrule",
    *[f"{label} & " + " & ".join(row[m] for m in metrics) + r" \\" for label, row in table_a7.iterrows()],
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
]
print("\n".join(lines))
write_output("table_a7.tex", "\n".join(lines))

\begin{table}[ht]
\centering
\caption{Model performance across 20 runs per condition based on electrode inclusion. Notation: Mean (std.). Two-sided Mann--Whitney $U$ test on balanced accuracy, all available electrodes vs.\ without facial MC electrodes (normal approximation with tie correction, no continuity correction): $U = 281$, $z = 2.19$, $p = .028$, $r = .35$.}
\label{tab:facial_mc_ablation}
\begin{tabular}{lccc}
\toprule
Condition & AUROC & Bal.\ Acc. & F1 \\
\midrule
All available electrodes & 0.725 (0.149) & 0.664 (0.116) & 0.663 (0.109) \\
Facial MC electrodes only & 0.688 (0.165) & 0.635 (0.120) & 0.622 (0.123) \\
Without facial MC electrodes & 0.612 (0.129) & 0.584 (0.099) & 0.587 (0.100) \\
\bottomrule
\end{tabular}
\end{table}
written: /viper/u1/$USER/projects/foundational_xai_in_neuroimaging/results/apa_mwu/table_a7.tex


## 7. Output files: statistics cache

`results/apa_mwu/table_a7_stats.json` holds the per-run test results (inputs), Table A7, the test result with all APA
quantities and both group descriptions, the comparison with the earlier p-value, the test settings, the git commit,
library versions and the folder the runs were read from.

In [9]:
stats_cache = {
    "description": "Table A7: PhysioMI L/R facial MC electrode ablation, finetuned LaBraM, 20 runs per condition",
    "provenance": provenance({"OUTPUT_DIR": OUTPUT_DIR,
                              **{c: os.path.realpath(os.path.join(OUTPUT_DIR, c)) for c in CONDITIONS}}),
    "inputs": {"runs": runs},
    "table_a7_mean_std": table_a7.reset_index(),
    "mann_whitney_u_balanced_accuracy": {**mwu, "group_x": describe_group(mwu, "x"), "group_y": describe_group(mwu, "y"),
                                         "apa": apa_mwu_text(mwu), "sentence": apa_mwu_sentence(mwu)},
    "comparison_with_earlier_version": comparison,
}
write_output("table_a7_stats.json", stats_cache)

written: /viper/u1/$USER/projects/foundational_xai_in_neuroimaging/results/apa_mwu/table_a7_stats.json
